# 01 DB Population

Purpose: Populate persistent ChromaDB image/text collections from training data (idempotent).

Inputs: `dataset/CVPR_2024_dataset_Train/`, `dataset_text/train.csv`.

Outputs: Persistent collections in `chroma_db/` (`trash_image_db`, `trash_text_db`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../..').resolve()))

import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from src.phase2.db_client import get_image_collection, get_persistent_client, get_text_collection

CONFIG = {
    'k_vote': 10,
    'K_density': 50,
    'alpha': 0.5,
    'alpha_sweep': [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0],
    'imbalance_ratios': [10, 50, 100],
    'batch_size': 100,
    'epsilon': 1e-6,
    'db_path': './chroma_db',
    'figures_path': './figures/phase2',
    'class_names': ['Black', 'Blue', 'Green', 'TTR'],
    'minority_classes': ['TTR', 'Green'],
    'majority_classes': ['Black', 'Blue'],
}

REPO_ROOT = Path('../..').resolve()
TRAIN_DIR = REPO_ROOT / 'dataset' / 'CVPR_2024_dataset_Train'
TRAIN_CSV = REPO_ROOT / 'dataset_text' / 'train.csv'

if not TRAIN_DIR.exists() or not TRAIN_CSV.exists():
    raise FileNotFoundError('Training dataset missing. Expected dataset and dataset_text paths in repo root.')

train_df = pd.read_csv(TRAIN_CSV)
if not {'text', 'label'}.issubset(train_df.columns):
    raise ValueError('train.csv must contain columns: text, label')

records = []
for row in train_df.itertuples(index=False):
    image_path = TRAIN_DIR / row.label / row.text
    records.append({'image_path': image_path, 'filename': row.text, 'label': row.label})

print(f'Training records: {len(records)}')

In [ ]:
try:
    client = get_persistent_client(str(REPO_ROOT / 'chroma_db'))
    image_collection = get_image_collection(client)
    text_collection = get_text_collection(client)
except Exception as exc:
    raise RuntimeError(f'ChromaDB initialization failed: {exc}') from exc

expected_count = len(records)
if image_collection.count() == expected_count and text_collection.count() == expected_count:
    print('Collections already populated. Skipping insertion.')
    SKIP_POPULATION = True
else:
    SKIP_POPULATION = False

In [ ]:
if not SKIP_POPULATION:
    batch_size = CONFIG['batch_size']
    for start in tqdm(range(0, len(records), batch_size), desc='Populating ChromaDB'):
        batch = records[start:start + batch_size]
        ids_img, ids_txt = [], []
        images, docs = [], []
        metadatas = []

        for idx, record in enumerate(batch, start=start):
            image = Image.open(record['image_path']).convert('RGB')
            image_np = np.array(image, dtype=np.uint8)
            ids_img.append(f'img_{idx}')
            ids_txt.append(f'txt_{idx}')
            images.append(image_np)
            docs.append(record['filename'])
            metadatas.append({'label': record['label'], 'filename': record['filename']})

        try:
            image_collection.add(ids=ids_img, images=images, metadatas=metadatas)
            text_collection.add(ids=ids_txt, documents=docs, metadatas=metadatas)
        except Exception as exc:
            raise RuntimeError(f'ChromaDB insert failed for batch starting at {start}: {exc}') from exc

print('Population step complete.')

In [ ]:
print(f"Image collection count: {image_collection.count()}")
print(f"Text collection count: {text_collection.count()}")